# One-time frozen predictive held-out test

Run all cells in a fresh Colab CPU or GPU runtime. Repository and dependency checks run before project imports. The held-out split remains locked until the exact confirmation phrase is entered. No training or checkpoint selection occurs here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

CONTENT_ROOT = Path('/content')
REPO_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR = CONTENT_ROOT / 'VitalDB-Feature-Selection'
DRIVE_PROJECT = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
REQUIRED_WORKFLOW_COMMIT = '16c2aab1f9ce04b4c375060026d3819a13d46aaf'

In [ ]:
import importlib
import os
import shlex
import shutil
import subprocess
import sys

def run_command(command, *, cwd=None, check=True, stream=False):
    command = [str(part) for part in command]
    print('$', shlex.join(command), flush=True)
    result = subprocess.run(
        command, cwd=cwd, check=False, capture_output=not stream, text=True
    )
    if result.stdout:
        print('[stdout]')
        print(result.stdout.rstrip())
    if result.stderr:
        print('[stderr]')
        print(result.stderr.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(
            f'Command failed with exit status {result.returncode}: {shlex.join(command)}\n'
            f'stdout:\n{result.stdout or "<streamed or empty>"}\n'
            f'stderr:\n{result.stderr or "<streamed or empty>"}'
        )
    return result

def verify_repository_state(repo_dir, required_ancestor):
    local_head = run_command(['git', '-C', repo_dir, 'rev-parse', 'HEAD']).stdout.strip()
    origin_head = run_command(['git', '-C', repo_dir, 'rev-parse', 'origin/main']).stdout.strip()
    remote_url = run_command(['git', '-C', repo_dir, 'remote', 'get-url', 'origin']).stdout.strip()
    branch_result = run_command(['git', '-C', repo_dir, 'branch', '--show-current'], check=False)
    branch = branch_result.stdout.strip() or '<detached>'
    if local_head != origin_head:
        raise RuntimeError(f'Repository is not synchronized: HEAD={local_head}, origin/main={origin_head}.')
    ancestor = run_command(
        ['git', '-C', repo_dir, 'merge-base', '--is-ancestor', required_ancestor, 'HEAD'],
        check=False,
    )
    if ancestor.returncode != 0:
        raise RuntimeError(f'Required workflow commit {required_ancestor} is not an ancestor of HEAD {local_head}.')
    print('Repository state:', {'HEAD': local_head, 'origin/main': origin_head, 'origin': remote_url, 'branch': branch})
    return {'head': local_head, 'origin_main': origin_head, 'origin_url': remote_url, 'branch': branch}

def synchronize_repository(repo_dir, repo_url, required_ancestor, *, allowed_parent):
    repo_dir = Path(repo_dir)
    allowed_parent = Path(allowed_parent).resolve()
    if repo_dir.parent.resolve() != allowed_parent:
        raise RuntimeError(f'Refusing repository operation outside {allowed_parent}: {repo_dir}')
    allowed_parent.mkdir(parents=True, exist_ok=True)
    os.chdir(allowed_parent)
    if (repo_dir / '.git').is_dir():
        current_remote = run_command(['git', '-C', repo_dir, 'remote', 'get-url', 'origin'], check=False)
        if current_remote.returncode != 0:
            run_command(['git', '-C', repo_dir, 'remote', 'add', 'origin', repo_url])
        elif current_remote.stdout.strip() != repo_url:
            run_command(['git', '-C', repo_dir, 'remote', 'set-url', 'origin', repo_url])
        run_command(['git', '-C', repo_dir, 'fetch', 'origin', 'main', '--prune'], stream=True)
        run_command(['git', '-C', repo_dir, 'reset', '--hard', 'origin/main'], stream=True)
    else:
        if repo_dir.exists() or repo_dir.is_symlink():
            print('Replacing non-Git temporary repository path:', repo_dir)
            if repo_dir.is_symlink():
                repo_dir.unlink()
            else:
                shutil.rmtree(repo_dir)
        run_command(['git', 'clone', repo_url, repo_dir], stream=True)
    return verify_repository_state(repo_dir, required_ancestor)

def prepare_project_imports(repo_dir, prefixes=('src', 'scripts')):
    stale = [name for name in list(sys.modules) if any(name == prefix or name.startswith(prefix + '.') for prefix in prefixes)]
    for name in stale:
        sys.modules.pop(name, None)
    if stale:
        print('Purged stale project modules:', sorted(stale))
    resolved_repo = Path(repo_dir).resolve()
    retained = []
    for entry in sys.path:
        try:
            if Path(entry or os.curdir).resolve() == resolved_repo:
                continue
        except OSError:
            pass
        retained.append(entry)
    sys.path[:] = [str(resolved_repo), *retained]
    importlib.invalidate_caches()
    return stale

REPOSITORY_STATE = synchronize_repository(REPO_DIR, REPO_URL, REQUIRED_WORKFLOW_COMMIT, allowed_parent=CONTENT_ROOT)
PURGED_PROJECT_MODULES = prepare_project_imports(REPO_DIR)
os.chdir(REPO_DIR)

In [ ]:
run_command(
    [sys.executable, '-u', 'scripts/install_colab_dependencies.py', '--profile', 'frozen-test'],
    cwd=REPO_DIR, stream=True,
)
prepare_project_imports(REPO_DIR)
from src.colab_workflow import frozen_test_runtime_versions
RUNTIME_VERSIONS = frozen_test_runtime_versions()
print('Frozen-test runtime:', RUNTIME_VERSIONS)

In [ ]:
import hashlib
import json
import pandas as pd
from IPython.display import display, Markdown

DATASET_DIR = DRIVE_PROJECT / 'data/modeling/full'
ANALYSIS_MANIFEST = DRIVE_PROJECT / 'outputs/frozen_candidate_retraining_validation_only/analysis/analysis_manifest.json'
STRICT_ROOT = DRIVE_PROJECT / 'outputs/frozen_candidate_retraining_validation_only/strict_consensus'
FULL17_ROOT = DRIVE_PROJECT / 'outputs/group_retraining_validation_only/full17'
DECISION_TEMPLATE_DIR = REPO_DIR / 'outputs/frozen_predictive_decision_30s'
DECISION_DIR = DRIVE_PROJECT / 'outputs/frozen_predictive_decision_30s'
OUTPUT_DIR = DRIVE_PROJECT / 'outputs/frozen_predictive_test_evaluation_30s'
HANDOFF_DIR = DRIVE_PROJECT / 'outputs/predictive_stage_handoff'
required = [DATASET_DIR, ANALYSIS_MANIFEST, STRICT_ROOT, FULL17_ROOT, DECISION_TEMPLATE_DIR]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required Drive/repository inputs: {missing}')
COMMON_ARGS = [
    '--decision-template-dir', str(DECISION_TEMPLATE_DIR), '--decision-dir', str(DECISION_DIR),
    '--analysis-manifest', str(ANALYSIS_MANIFEST), '--dataset-dir', str(DATASET_DIR),
    '--strict-root', str(STRICT_ROOT), '--full17-root', str(FULL17_ROOT), '--output-dir', str(OUTPUT_DIR),
]
run_command([sys.executable, '-u', 'scripts/evaluate_frozen_predictive_test.py', *COMMON_ARGS, '--preflight-only'], cwd=REPO_DIR, stream=True)
decision_path = OUTPUT_DIR / 'frozen_decision_snapshot.json'
inventory_path = OUTPUT_DIR / 'evaluated_checkpoint_inventory.csv'
decision = json.loads(decision_path.read_text())
inventory = pd.read_csv(inventory_path)
decision_sha = hashlib.sha256(decision_path.read_bytes()).hexdigest()
assert decision['primary_candidate'] == 'strict_consensus'
assert decision['test_evaluation_candidates'] == ['strict_consensus', 'full17_reference']
assert 'compact_consensus' not in set(inventory['candidate'])
assert len(inventory) == 20 and not inventory.duplicated(['candidate', 'model', 'seed']).any()
assert set(inventory['checkpoint_name']) == {'best_model.pt'}
assert inventory['run_complete'].eq(True).all()
assert inventory['run_test_evaluated'].eq(False).all()
assert inventory['evaluate_test'].eq(False).all()
assert inventory['inference_only'].eq(True).all()
workflow_source = (REPO_DIR / 'src/frozen_predictive_test_evaluation.py').read_text()
for forbidden in ('run_gru_training', 'run_attention_training', 'torch.optim', '.backward('):
    assert forbidden not in workflow_source, forbidden
print('Active commit:', REPOSITORY_STATE['head'])
print('Dataset fingerprint:', inventory['dataset_fingerprint'].iloc[0])
print('Frozen decision manifest SHA256:', decision_sha)
print('Primary:', decision['primary_candidate'], decision['primary_dynamic_feature_names'])
print('Reference:', decision['reference_candidate'], decision['reference_dynamic_feature_names'])
print('compact_consensus held-out test excluded:', 'compact_consensus' not in set(inventory['candidate']))
display(inventory[['candidate', 'model', 'seed', 'checkpoint_path', 'checkpoint_sha256', 'run_complete', 'run_test_evaluated', 'evaluate_test']])
display(Markdown('**Preflight complete. The held-out split has not been opened.**'))

In [ ]:
run_command(
    [sys.executable, '-u', 'scripts/build_predictive_stage_handoff.py',
     '--repo-dir', str(REPO_DIR), '--dataset-dir', str(DATASET_DIR),
     '--decision', str(decision_path), '--checkpoint-inventory', str(inventory_path),
     '--output-dir', str(HANDOFF_DIR)],
    cwd=REPO_DIR, stream=True,
)
display(json.loads((HANDOFF_DIR / 'predictive_stage_manifest.json').read_text()))

In [ ]:
CONFIRMATION = 'RUN_ONE_TIME_FROZEN_PREDICTIVE_TEST'
entered = input(f'Type {CONFIRMATION} to perform the one-time held-out inference: ').strip()
if entered != CONFIRMATION:
    raise RuntimeError('Confirmation mismatch. Held-out inference was not started.')

In [ ]:
run_command(
    [sys.executable, '-u', 'scripts/evaluate_frozen_predictive_test.py', *COMMON_ARGS, '--device', 'auto', '--confirmation', entered],
    cwd=REPO_DIR, stream=True,
)
print('One-time frozen evaluation command completed.')

In [ ]:
for name in [
    'test_candidate_aggregate.csv', 'paired_test_seed_deltas.csv',
    'paired_model_test_contrasts.csv', 'hierarchical_bootstrap_test_contrasts.csv',
]:
    display(Markdown(f'## {name}'))
    display(pd.read_csv(OUTPUT_DIR / name))
display(Markdown((OUTPUT_DIR / 'frozen_predictive_test_report.md').read_text()))
display(json.loads((OUTPUT_DIR / 'test_evaluation_manifest.json').read_text()))